# Serve ESOL finetune via HTTP API

> Local: load the fixed ESOL checkpoint, then call embeddings / predictions.
> Railway: set `API` to your public URL — weights auto-load from Hugging Face (skip `/load`).

**Local checkpoint:**
`finetune_ckpts/esol/smi-ted-Light-Finetune_seed0_esol_epoch=3_valloss=0.7972.pt`

Deploy notes: [`docs/railway.md`](../docs/railway.md).

Start the API locally (separate terminal):

```bash
cd max_ports && pixi run api
# → http://127.0.0.1:8080
```

| Method | Path | Purpose |
|--------|------|---------|
| `POST` | `/load` | Export + serve the ESOL finetune (local only) |
| `POST` | `/embeddings` | SMILES → embeddings + predictions |
| `GET` | `/status` | Is MAX ready? |
| `POST` | `/stop` | Stop MAX Serve |

In [ ]:
from __future__ import annotations

import json
from typing import Any

import httpx

API = "http://127.0.0.1:8080"  # or https://….up.railway.app

CHECKPOINT = (
    "./finetune_ckpts/esol/"
    "smi-ted-Light-Finetune_seed0_esol_epoch=3_valloss=0.7972.pt"
)


def api(method: str, path: str, **kwargs) -> Any:
    r = httpx.request(method, f"{API.rstrip('/')}{path}", timeout=300.0, **kwargs)
    if r.status_code >= 400:
        raise RuntimeError(f"{r.status_code}: {r.text}")
    return r.json()


print(api("GET", "/health"))
print(json.dumps(api("GET", "/status"), indent=2))

## Load ESOL finetune (local only)

Skip this cell on Railway — `MATGRAM_AUTO_LOAD=1` already loaded the HF assets.

Locally, exports the checkpoint into `model_assets/smi-ted-esol/` and starts MAX Serve.

In [ ]:
status = api(
    "POST",
    "/load",
    json={
        "checkpoint": CHECKPOINT,
        "task": "esol",
        "device": "cpu",  # or "gpu"
    },
)
print(json.dumps(status, indent=2))

## Embeddings / predictions

Property mode fills both `embeddings` and `predictions` (ESOL solubility).

In [ ]:
SMILES = ["CCO", "c1ccccc1", "CC(=O)O"]
out = api("POST", "/embeddings", json={"smiles": SMILES})
print("mode:", out["mode"], "task:", out["task"], "model:", out["model"])
for s, pred, vec in zip(SMILES, out["predictions"], out["embeddings"]):
    print(f"{s:12s}  prediction={pred:.6f}  dim={len(vec)}")

## curl equivalents

```bash
curl -s http://127.0.0.1:8080/status | jq

# local only — not needed on Railway
curl -s -X POST http://127.0.0.1:8080/load \
  -H 'Content-Type: application/json' \
  -d '{"checkpoint":"./finetune_ckpts/esol/smi-ted-Light-Finetune_seed0_esol_epoch=3_valloss=0.7972.pt","task":"esol","device":"cpu"}' | jq

curl -s -X POST http://127.0.0.1:8080/embeddings \
  -H 'Content-Type: application/json' \
  -d '{"smiles":"CCO"}' | jq
```

In [ ]:
# api("POST", "/stop")  # optional: free GPU/CPU when done